# Capstone Project: AI Workflow with LangChain & LangGraph

This notebook demonstrates a practical capstone project using [LangChain](https://python.langchain.com/) and [LangGraph](https://langchain-ai.github.io/langgraph/) in Google Colab. You will:
- Set up the environment
- Load and prepare data
- Build LangChain components
- Construct a LangGraph workflow
- Execute and evaluate the workflow

**Goal:** Create an end-to-end AI workflow (e.g., document QA or RAG) using modern agentic tools.

## 1. Set Up Environment and Install Dependencies

Run the following cell to install LangChain, LangGraph, and other required libraries. This is designed for Google Colab.

In [ ]:
# Install LangChain, LangGraph, and other dependencies
!pip install --quiet langchain langgraph openai tiktoken pandas matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


## 2. Import Required Libraries

Import all necessary Python libraries for the project.

In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
import langgraph
from langgraph.graph import StateGraph, END

# Set your OpenAI API key (for demo, use environment variable or prompt user)
os.environ["OPENAI_API_KEY"] = "sk-..."  # Replace with your key or use Colab secrets

ModuleNotFoundError: No module named 'langchain.llms'

## 3. Load and Prepare Data

For this demo, we'll use a small sample dataset. You can replace this with your own data for the capstone project.

In [ ]:
# Example: Load a small dataset (replace with your own for capstone)
data = pd.DataFrame({
    'question': [
        'What is LangChain?',
        'How does LangGraph work?',
        'What is an LLM?'
    ],
    'context': [
        'LangChain is a framework for developing applications powered by language models.',
        'LangGraph enables graph-based workflows for LLM applications.',
        'LLM stands for Large Language Model.'
    ]
})
data

## 4. Define LangChain Components

We'll define a simple prompt template and an LLM chain for question answering using the context.

In [ ]:
# Define a prompt template for QA
template = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use the following context to answer the question.\nContext: {context}\nQuestion: {question}\nAnswer:"""
)

# Create an LLM chain (OpenAI, can swap for other providers)
llm = OpenAI(temperature=0)
qa_chain = LLMChain(llm=llm, prompt=template)

## 5. Build a LangGraph Workflow

We'll use LangGraph to create a simple workflow that applies the QA chain to each row in the dataset.

In [ ]:
# Define a simple LangGraph workflow for batch QA
class QAState:
    def __init__(self, data):
        self.data = data
        self.answers = []

def qa_node(state: QAState):
    for idx, row in state.data.iterrows():
        answer = qa_chain.run({"context": row["context"], "question": row["question"]})
        state.answers.append({"question": row["question"], "answer": answer})
    return state

# Build the graph
graph = StateGraph(QAState)
graph.add_node("qa", qa_node)
graph.set_entry_point("qa")
graph.set_finish_point(END)
workflow = graph.compile()

## 6. Execute the Capstone Workflow

Run the workflow on the prepared data and collect the results.

In [ ]:
# Run the workflow
state = QAState(data)
result_state = workflow(state)
answers = pd.DataFrame(result_state.answers)
answers

## 7. Evaluate and Visualize Results

Let's visualize the results and discuss the workflow's effectiveness.

In [ ]:
# Visualize the results
def plot_answers(df):
    fig, ax = plt.subplots(figsize=(8, 2 * len(df)))
    ax.axis('off')
    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc='left',
        loc='center',
        colColours=["#f2f2f2"]*len(df.columns)
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.auto_set_column_width(col=list(range(len(df.columns))))
    plt.show()

plot_answers(answers)

# Discussion: Review the answers and consider how to improve the workflow (e.g., better prompts, more data, advanced graph logic).